In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("환경 세팅 완료")

pandas: 3.0.3
numpy: 2.4.6
환경 세팅 완료


In [6]:
import pandas as pd

url = "https://m.sports.naver.com/kbaseball/record/kbo?seasonCode=2026&tab=hitter"

try:
    tables = pd.read_html(url)
    print("가져온 표 개수:", len(tables))

    for i, table in enumerate(tables):
        print(f"\n--- {i}번째 표 ---")
        display(table.head())

except Exception as e:
    print("오류 발생:")
    print(e)

오류 발생:
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>


In [9]:
import requests
from bs4 import BeautifulSoup
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://m.sports.naver.com/kbaseball/record/kbo?seasonCode=2026&tab=hitter"

response = requests.get(url, verify=False)
html = response.text

print("상태코드:", response.status_code)
print("HTML 길이:", len(html))

soup = BeautifulSoup(html, "html.parser")

tables = soup.find_all("table")
print("table 개수:", len(tables))

print("WAR 포함 여부:", "WAR" in html)
print("wRC+ 포함 여부:", "wRC+" in html)
print("OPS 포함 여부:", "OPS" in html)

상태코드: 200
HTML 길이: 1842
table 개수: 0
WAR 포함 여부: False
wRC+ 포함 여부: False
OPS 포함 여부: False


In [11]:
import requests
import pandas as pd
import urllib3

# SSL 경고 숨기기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 네이버 KBO 타자 기록 API
url = "https://api-gw.sports.naver.com/statistics/categories/kbo/seasons/2026/players"

params = {
    "sortField": "hitterHra",
    "sortDirection": "desc",
    "playerType": "HITTER",
    "gameType": "REGULAR_SEASON"
}

headers = {
    "accept": "application/json",
    "origin": "https://m.sports.naver.com",
    "referer": "https://m.sports.naver.com/kbaseball/record/kbo?seasonCode=2026&tab=hitter",
    "user-agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    verify=False
)

print("상태코드:", response.status_code)

data = response.json()

players = data["result"]["seasonPlayerStats"]

print("가져온 선수 수:", len(players))

df = pd.DataFrame(players)

print("데이터 크기:", df.shape)

display(df.head())

df.to_csv("../data/naver_hitter_2026_raw.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ../data/naver_hitter_2026_raw.csv")

상태코드: 200
가져온 선수 수: 50
데이터 크기: (50, 48)


,ranking,playerId,playerName,playerImageUrl,weight,height,backNumber,isRetire,isPlayer,osId,...,hitterSlg,hitterOps,hitterIsop,hitterBabip,hitterWoba,hitterWrcPlus,hitterWpa,hitterWar,isQualified,isKoreanPlayer
0,1.0,66606,최원준,https://sports-phinf.pstatic.net/player/kbo/de...,85,178,3,N,Y,2941437,...,0.533,0.994,0.150,0.440,0.437,169.0,2.794,3.68,True,False
1,2.0,54529,레이예스,https://sports-phinf.pstatic.net/player/kbo/de...,87,196,29,N,Y,26248643,...,0.547,0.968,0.187,0.382,0.425,149.8,2.150,3.21,True,False
2,3.0,53123,오스틴,https://sports-phinf.pstatic.net/player/kbo/de...,97,183,23,N,Y,18107881,...,0.667,1.095,0.311,0.363,0.469,199.6,3.424,4.64,True,False
3,4.0,67893,박성한,https://sports-phinf.pstatic.net/player/kbo/de...,77,180,2,N,Y,3999802,...,0.467,0.924,0.113,0.390,0.429,165.1,1.914,3.76,True,False
4,5.0,63260,이우성,https://sports-phinf.pstatic.net/player/kbo/de...,95,182,55,N,Y,285855,...,0.464,0.846,0.119,0.398,0.386,136.3,0.205,1.70,True,False


저장 완료: ../data/naver_hitter_2026_raw.csv


In [13]:
import requests
import pandas as pd
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

base_url = "https://api-gw.sports.naver.com/statistics/categories/kbo/seasons/{year}/players"

headers = {
    "accept": "application/json",
    "origin": "https://m.sports.naver.com",
    "referer": "https://m.sports.naver.com/kbaseball/record/kbo",
    "user-agent": "Mozilla/5.0"
}

all_hitter_data = []

for year in range(2018, 2027):
    print(f"\n===== {year} 시즌 수집 시작 =====")

    for page in range(1, 6):  # 연도별 최대 5페이지까지만
        params = {
            "page": page,
            "pageSize": 50,
            "sortField": "hitterWar",
            "sortDirection": "desc",
            "playerType": "HITTER",
            "gameType": "REGULAR_SEASON"
        }

        url = base_url.format(year=year)

        response = requests.get(url, params=params, headers=headers, verify=False)
        data = response.json()
        players = data["result"].get("seasonPlayerStats", [])

        print(f"{year} page {page}: {len(players)}명")

        if not players:
            break

        for player in players:
            player["collect_year"] = year
            player["page"] = page
            all_hitter_data.append(player)

        time.sleep(0.3)

hitter_raw_all = pd.DataFrame(all_hitter_data)

# 중복 제거: 같은 연도-선수ID가 반복되면 하나만 남김
hitter_raw_all = hitter_raw_all.drop_duplicates(subset=["collect_year", "playerId"])

print("전체 데이터 크기:", hitter_raw_all.shape)
display(hitter_raw_all.head())

hitter_raw_all.to_csv("../data/naver_hitter_2018_2026_raw.csv", index=False, encoding="utf-8-sig")
print("저장 완료")


===== 2018 시즌 수집 시작 =====
2018 page 1: 50명
2018 page 2: 50명
2018 page 3: 50명
2018 page 4: 50명
2018 page 5: 50명

===== 2019 시즌 수집 시작 =====
2019 page 1: 50명
2019 page 2: 50명
2019 page 3: 50명
2019 page 4: 50명
2019 page 5: 50명

===== 2020 시즌 수집 시작 =====
2020 page 1: 50명
2020 page 2: 50명
2020 page 3: 50명
2020 page 4: 50명
2020 page 5: 50명

===== 2021 시즌 수집 시작 =====
2021 page 1: 50명
2021 page 2: 50명
2021 page 3: 50명
2021 page 4: 50명
2021 page 5: 50명

===== 2022 시즌 수집 시작 =====
2022 page 1: 50명
2022 page 2: 50명
2022 page 3: 50명
2022 page 4: 50명
2022 page 5: 50명

===== 2023 시즌 수집 시작 =====
2023 page 1: 50명
2023 page 2: 50명
2023 page 3: 50명
2023 page 4: 50명
2023 page 5: 50명

===== 2024 시즌 수집 시작 =====
2024 page 1: 50명
2024 page 2: 50명
2024 page 3: 50명
2024 page 4: 50명
2024 page 5: 50명

===== 2025 시즌 수집 시작 =====
2025 page 1: 50명
2025 page 2: 50명
2025 page 3: 50명
2025 page 4: 50명
2025 page 5: 50명

===== 2026 시즌 수집 시작 =====
2026 page 1: 50명
2026 page 2: 50명
2026 page 3: 50명
2026 page 4: 50명
2026 page

,ranking,playerId,playerName,playerImageUrl,weight,height,backNumber,isRetire,isPlayer,osId,...,hitterIsop,hitterBabip,hitterWoba,hitterWrcPlus,hitterWpa,hitterWar,isQualified,isKoreanPlayer,collect_year,page
0,None,78224,김재환,https://sports-phinf.pstatic.net/player/kbo/de...,98,184,32,N,Y,154249,...,0.323,0.371,0.450,184.9,3.839,9.39,False,False,2018,1
1,None,75125,박병호,https://sports-phinf.pstatic.net/player/kbo/de...,107,185,52,Y,Y,104972,...,0.373,0.386,0.490,169.3,5.250,7.18,False,False,2018,1
2,None,67450,러프,https://sports-phinf.pstatic.net/player/kbo/de...,113,190,50,N,Y,552268,...,0.275,0.358,0.443,141.8,4.102,7.12,False,False,2018,1
3,None,76267,최주환,https://sports-phinf.pstatic.net/player/kbo/de...,73,177,53,N,Y,124615,...,0.249,0.355,0.423,162.2,2.592,6.58,False,False,2018,1
4,None,79192,채은성,https://sports-phinf.pstatic.net/player/kbo/de...,92,186,22,N,Y,217571,...,0.217,0.364,0.399,161.2,2.205,6.52,False,False,2018,1


저장 완료


In [14]:
import requests
import pandas as pd
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

base_url = "https://api-gw.sports.naver.com/statistics/categories/kbo/seasons/{year}/players"

headers = {
    "accept": "application/json",
    "origin": "https://m.sports.naver.com",
    "referer": "https://m.sports.naver.com/kbaseball/record/kbo",
    "user-agent": "Mozilla/5.0"
}

years = range(2018, 2027)

sort_fields = [
    "hitterWar",
    "hitterOps",
    "hitterWrcPlus",
    "hitterHra",
    "hitterHr",
    "hitterRbi",
    "hitterGameCount",
    "hitterSb",
    "hitterObp",
    "hitterSlg",
    "hitterWpa",
    "hitterBb",
    "hitterHit"
]

all_data = []
seen = set()

for year in years:
    print(f"\n===== {year} 시즌 수집 시작 =====")

    for sort_field in sort_fields:
        params = {
            "sortField": sort_field,
            "sortDirection": "desc",
            "playerType": "HITTER",
            "gameType": "REGULAR_SEASON"
        }

        url = base_url.format(year=year)

        response = requests.get(
            url,
            params=params,
            headers=headers,
            verify=False
        )

        if response.status_code != 200:
            print(f"{year} / {sort_field}: 요청 실패 {response.status_code}")
            continue

        data = response.json()
        players = data.get("result", {}).get("seasonPlayerStats", [])

        new_count = 0

        for player in players:
            if player.get("isKoreanPlayer") != "Y":
                continue

            key = (year, player.get("playerId"))

            if key in seen:
                continue

            seen.add(key)

            player["collect_year"] = year
            player["sort_source"] = sort_field

            all_data.append(player)
            new_count += 1

        print(f"{year} / {sort_field}: 신규 국내 선수 {new_count}명")

        time.sleep(0.3)

hitter_raw = pd.DataFrame(all_data)

print("\n원천 데이터 크기:", hitter_raw.shape)

hitter_raw.to_csv(
    "../data/naver_hitter_2018_2026_korean_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

selected_cols = [
    "collect_year",
    "playerId",
    "playerName",
    "teamName",
    "isRetire",
    "isPlayer",
    "isKoreanPlayer",

    "hitterGameCount",
    "hitterAb",

    "hitterHra",
    "hitterObp",
    "hitterSlg",
    "hitterOps",
    "hitterWrcPlus",
    "hitterHr",
    "hitterRbi",
    "hitterRun",
    "hitterHit",
    "hitterH2",
    "hitterH3",
    "hitterBb",
    "hitterKk",

    "hitterSb",
    "hitterCs",

    "hitterGd",
    "hitterWpa",
    "hitterWar",

    "sort_source"
]

selected_cols = [col for col in selected_cols if col in hitter_raw.columns]

hitter_selected = hitter_raw[selected_cols].copy()

hitter_selected.to_csv(
    "../data/hitter_season_stats_2018_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("선택 컬럼 저장 완료: ../data/hitter_season_stats_2018_2026.csv")
print("최종 선택 데이터 크기:", hitter_selected.shape)

display(hitter_selected.head())


===== 2018 시즌 수집 시작 =====
2018 / hitterWar: 신규 국내 선수 0명
2018 / hitterOps: 신규 국내 선수 0명
2018 / hitterWrcPlus: 신규 국내 선수 0명
2018 / hitterHra: 신규 국내 선수 0명
2018 / hitterHr: 신규 국내 선수 0명
2018 / hitterRbi: 신규 국내 선수 0명
2018 / hitterGameCount: 신규 국내 선수 0명
2018 / hitterSb: 신규 국내 선수 0명
2018 / hitterObp: 신규 국내 선수 0명
2018 / hitterSlg: 신규 국내 선수 0명
2018 / hitterWpa: 신규 국내 선수 0명
2018 / hitterBb: 신규 국내 선수 0명
2018 / hitterHit: 신규 국내 선수 0명

===== 2019 시즌 수집 시작 =====
2019 / hitterWar: 신규 국내 선수 0명
2019 / hitterOps: 신규 국내 선수 0명
2019 / hitterWrcPlus: 신규 국내 선수 0명
2019 / hitterHra: 신규 국내 선수 0명
2019 / hitterHr: 신규 국내 선수 0명
2019 / hitterRbi: 신규 국내 선수 0명
2019 / hitterGameCount: 신규 국내 선수 0명
2019 / hitterSb: 신규 국내 선수 0명
2019 / hitterObp: 신규 국내 선수 0명
2019 / hitterSlg: 신규 국내 선수 0명
2019 / hitterWpa: 신규 국내 선수 0명
2019 / hitterBb: 신규 국내 선수 0명
2019 / hitterHit: 신규 국내 선수 0명

===== 2020 시즌 수집 시작 =====
2020 / hitterWar: 신규 국내 선수 0명
2020 / hitterOps: 신규 국내 선수 0명
2020 / hitterWrcPlus: 신규 국내 선수 0명
2020 / hitterHra: 신규 국내 선수 0명
2

""


In [15]:
import requests
import pandas as pd
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

base_url = "https://api-gw.sports.naver.com/statistics/categories/kbo/seasons/{year}/players"

headers = {
    "accept": "application/json",
    "origin": "https://m.sports.naver.com",
    "referer": "https://m.sports.naver.com/kbaseball/record/kbo",
    "user-agent": "Mozilla/5.0"
}

years = range(2018, 2027)

sort_fields = [
    "hitterWar",
    "hitterOps",
    "hitterWrcPlus",
    "hitterHra",
    "hitterHr",
    "hitterRbi",
    "hitterGameCount",
    "hitterSb",
    "hitterObp",
    "hitterSlg",
    "hitterWpa",
    "hitterBb",
    "hitterHit"
]

all_data = []
seen = set()

for year in years:
    print(f"\n===== {year} 시즌 수집 시작 =====")

    for sort_field in sort_fields:
        params = {
            "sortField": sort_field,
            "sortDirection": "desc",
            "playerType": "HITTER",
            "gameType": "REGULAR_SEASON"
        }

        url = base_url.format(year=year)

        response = requests.get(
            url,
            params=params,
            headers=headers,
            verify=False
        )

        if response.status_code != 200:
            print(f"{year} / {sort_field}: 요청 실패 {response.status_code}")
            continue

        data = response.json()
        players = data.get("result", {}).get("seasonPlayerStats", [])

        new_count = 0

        for player in players:
            key = (year, player.get("playerId"))

            if key in seen:
                continue

            seen.add(key)

            player["collect_year"] = year
            player["sort_source"] = sort_field

            all_data.append(player)
            new_count += 1

        print(f"{year} / {sort_field}: 신규 선수 {new_count}명")

        time.sleep(0.3)

hitter_raw = pd.DataFrame(all_data)

print("\n원천 데이터 크기:", hitter_raw.shape)

display(hitter_raw.head())

hitter_raw.to_csv(
    "../data/naver_hitter_2018_2026_raw_all.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료: ../data/naver_hitter_2018_2026_raw_all.csv")


===== 2018 시즌 수집 시작 =====
2018 / hitterWar: 신규 선수 50명
2018 / hitterOps: 신규 선수 5명
2018 / hitterWrcPlus: 신규 선수 5명
2018 / hitterHra: 신규 선수 2명
2018 / hitterHr: 신규 선수 7명
2018 / hitterRbi: 신규 선수 0명
2018 / hitterGameCount: 신규 선수 7명
2018 / hitterSb: 신규 선수 14명
2018 / hitterObp: 신규 선수 0명
2018 / hitterSlg: 신규 선수 0명
2018 / hitterWpa: 신규 선수 8명
2018 / hitterBb: 신규 선수 3명
2018 / hitterHit: 신규 선수 0명

===== 2019 시즌 수집 시작 =====
2019 / hitterWar: 신규 선수 50명
2019 / hitterOps: 신규 선수 7명
2019 / hitterWrcPlus: 신규 선수 1명
2019 / hitterHra: 신규 선수 1명
2019 / hitterHr: 신규 선수 11명
2019 / hitterRbi: 신규 선수 1명
2019 / hitterGameCount: 신규 선수 7명
2019 / hitterSb: 신규 선수 15명
2019 / hitterObp: 신규 선수 0명
2019 / hitterSlg: 신규 선수 0명
2019 / hitterWpa: 신규 선수 26명
2019 / hitterBb: 신규 선수 0명
2019 / hitterHit: 신규 선수 0명

===== 2020 시즌 수집 시작 =====
2020 / hitterWar: 신규 선수 50명
2020 / hitterOps: 신규 선수 8명
2020 / hitterWrcPlus: 신규 선수 0명
2020 / hitterHra: 신규 선수 1명
2020 / hitterHr: 신규 선수 8명
2020 / hitterRbi: 신규 선수 1명
2020 / hitterGameCount: 신규 선수 8

,ranking,playerId,playerName,playerImageUrl,weight,height,backNumber,isRetire,isPlayer,osId,...,hitterIsop,hitterBabip,hitterWoba,hitterWrcPlus,hitterWpa,hitterWar,isQualified,isKoreanPlayer,collect_year,sort_source
0,NaN,78224,김재환,https://sports-phinf.pstatic.net/player/kbo/de...,98,184,32,N,Y,154249,...,0.323,0.371,0.450,184.9,3.839,9.39,False,False,2018,hitterWar
1,NaN,75125,박병호,https://sports-phinf.pstatic.net/player/kbo/de...,107,185,52,Y,Y,104972,...,0.373,0.386,0.490,169.3,5.250,7.18,False,False,2018,hitterWar
2,NaN,67450,러프,https://sports-phinf.pstatic.net/player/kbo/de...,113,190,50,N,Y,552268,...,0.275,0.358,0.443,141.8,4.102,7.12,False,False,2018,hitterWar
3,NaN,76267,최주환,https://sports-phinf.pstatic.net/player/kbo/de...,73,177,53,N,Y,124615,...,0.249,0.355,0.423,162.2,2.592,6.58,False,False,2018,hitterWar
4,NaN,79192,채은성,https://sports-phinf.pstatic.net/player/kbo/de...,92,186,22,N,Y,217571,...,0.217,0.364,0.399,161.2,2.205,6.52,False,False,2018,hitterWar


저장 완료: ../data/naver_hitter_2018_2026_raw_all.csv


In [16]:
print("isKoreanPlayer 값 종류:")
print(hitter_raw["isKoreanPlayer"].unique())

print("isRetire 값 종류:")
print(hitter_raw["isRetire"].unique())

print("isPlayer 값 종류:")
print(hitter_raw["isPlayer"].unique())

isKoreanPlayer 값 종류:
[False]
isRetire 값 종류:
<StringArray>
['N', 'Y']
Length: 2, dtype: str
isPlayer 값 종류:
<StringArray>
['Y']
Length: 1, dtype: str


In [17]:
korean_values = ["Y", "y", "true", "True", True, 1, "1"]

hitter_korean = hitter_raw[
    hitter_raw["isKoreanPlayer"].isin(korean_values)
].copy()

print("국내 선수 필터링 후 데이터 크기:", hitter_korean.shape)
display(hitter_korean.head())

국내 선수 필터링 후 데이터 크기: (0, 50)


,ranking,playerId,playerName,playerImageUrl,weight,height,backNumber,isRetire,isPlayer,osId,...,hitterIsop,hitterBabip,hitterWoba,hitterWrcPlus,hitterWpa,hitterWar,isQualified,isKoreanPlayer,collect_year,sort_source


In [18]:
hitter_korean.to_csv(
    "../data/naver_hitter_2018_2026_korean_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료: ../data/naver_hitter_2018_2026_korean_raw.csv")

저장 완료: ../data/naver_hitter_2018_2026_korean_raw.csv


In [20]:
print("isKoreanPlayer 값 종류:")
print(hitter_raw["isKoreanPlayer"].unique())

isKoreanPlayer 값 종류:
[False]


In [21]:
print(hitter_raw[[
    "playerName",
    "teamName",
    "isKoreanPlayer"
]].head(30))

   playerName teamName  isKoreanPlayer
0         김재환       두산           False
1         박병호       넥센           False
2          러프       삼성           False
3         최주환       두산           False
4         채은성       LG           False
5         김현수       LG           False
6          로맥       SK           False
7         양의지       두산           False
8         손아섭       롯데           False
9         로하스       KT           False
10        나성범       NC           False
11        전준우       롯데           False
12        최형우      KIA           False
13        이대호       롯데           False
14        안치홍      KIA           False
15        오재일       두산           False
16        구자욱       삼성           False
17        한유섬       SK           False
18        김헌곤       삼성           False
19        이형종       LG           False
20        이정후       넥센           False
21         호잉       한화           False
22       버나디나      KIA           False
23        강백호       KT           False
24        노수광       SK   

In [22]:
selected_cols = [
    "collect_year",
    "playerId",
    "playerName",
    "teamName",
    "isRetire",
    "isPlayer",

    "hitterGameCount",
    "hitterAb",
    "hitterHra",
    "hitterObp",
    "hitterSlg",
    "hitterOps",
    "hitterWrcPlus",
    "hitterHr",
    "hitterRbi",
    "hitterRun",
    "hitterHit",
    "hitterH2",
    "hitterH3",
    "hitterBb",
    "hitterKk",
    "hitterSb",
    "hitterCs",
    "hitterGd",
    "hitterWpa",
    "hitterWar",
    "sort_source"
]

selected_cols = [col for col in selected_cols if col in hitter_raw.columns]

hitter_selected = hitter_raw[selected_cols].copy()

hitter_selected.to_csv(
    "../data/hitter_season_stats_2018_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료: ../data/hitter_season_stats_2018_2026.csv")
print("데이터 크기:", hitter_selected.shape)

display(hitter_selected.head())

저장 완료: ../data/hitter_season_stats_2018_2026.csv
데이터 크기: (1003, 27)


,collect_year,playerId,playerName,teamName,isRetire,isPlayer,hitterGameCount,hitterAb,hitterHra,hitterObp,...,hitterH2,hitterH3,hitterBb,hitterKk,hitterSb,hitterCs,hitterGd,hitterWpa,hitterWar,sort_source
0,2018,78224,김재환,두산,N,Y,139,527,0.333966,0.405,...,36,1,59,134,2,None,None,3.839,9.39,hitterWar
1,2018,75125,박병호,넥센,Y,Y,113,400,0.345000,0.457,...,20,0,68,114,0,None,None,5.250,7.18,hitterWar
2,2018,67450,러프,삼성,N,Y,137,506,0.330040,0.419,...,32,4,65,107,5,None,None,4.102,7.12,hitterWar
3,2018,76267,최주환,두산,N,Y,138,519,0.333333,0.397,...,39,6,51,89,1,None,None,2.592,6.58,hitterWar
4,2018,79192,채은성,LG,N,Y,139,529,0.330813,0.379,...,36,2,35,97,3,None,None,2.205,6.52,hitterWar


In [23]:
import requests
import pandas as pd
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

base_url = "https://api-gw.sports.naver.com/statistics/categories/kbo/seasons/{year}/players"

headers = {
    "accept": "application/json",
    "origin": "https://m.sports.naver.com",
    "referer": "https://m.sports.naver.com/kbaseball/record/kbo",
    "user-agent": "Mozilla/5.0"
}

years = range(2018, 2027)

# 투수는 평가 기준이 선발/불펜에 따라 달라서 여러 기준으로 가져옴
sort_fields = [
    "pitcherWar",          # 투수 종합 기여도
    "pitcherEra",          # 평균자책점
    "pitcherWhip",         # 이닝당 출루허용
    "pitcherInning",       # 이닝
    "pitcherWin",          # 승
    "pitcherSave",         # 세이브
    "pitcherHold",         # 홀드
    "pitcherStrikeOut",    # 탈삼진
    "pitcherGameCount"     # 등판 경기수
]

all_pitcher_data = []
seen = set()

for year in years:
    print(f"\n===== {year} 시즌 투수 수집 시작 =====")

    for sort_field in sort_fields:
        params = {
            "sortField": sort_field,
            "sortDirection": "desc",
            "playerType": "PITCHER",
            "gameType": "REGULAR_SEASON"
        }

        url = base_url.format(year=year)

        response = requests.get(
            url,
            params=params,
            headers=headers,
            verify=False
        )

        if response.status_code != 200:
            print(f"{year} / {sort_field}: 요청 실패 {response.status_code}")
            continue

        data = response.json()
        players = data.get("result", {}).get("seasonPlayerStats", [])

        new_count = 0

        for player in players:
            key = (year, player.get("playerId"))

            if key in seen:
                continue

            seen.add(key)

            player["collect_year"] = year
            player["sort_source"] = sort_field

            all_pitcher_data.append(player)
            new_count += 1

        print(f"{year} / {sort_field}: 신규 투수 {new_count}명")

        time.sleep(0.3)

pitcher_raw = pd.DataFrame(all_pitcher_data)

print("\n투수 원천 데이터 크기:", pitcher_raw.shape)

pitcher_raw.to_csv(
    "../data/naver_pitcher_2018_2026_raw_all.csv",
    index=False,
    encoding="utf-8-sig"
)

print("투수 원본 저장 완료: ../data/naver_pitcher_2018_2026_raw_all.csv")

display(pitcher_raw.head())


===== 2018 시즌 투수 수집 시작 =====
2018 / pitcherWar: 신규 투수 50명
2018 / pitcherEra: 신규 투수 28명
2018 / pitcherWhip: 신규 투수 6명
2018 / pitcherInning: 신규 투수 13명
2018 / pitcherWin: 신규 투수 4명
2018 / pitcherSave: 신규 투수 35명
2018 / pitcherHold: 신규 투수 18명
2018 / pitcherStrikeOut: 신규 투수 0명
2018 / pitcherGameCount: 신규 투수 1명

===== 2019 시즌 투수 수집 시작 =====
2019 / pitcherWar: 신규 투수 50명
2019 / pitcherEra: 신규 투수 25명
2019 / pitcherWhip: 신규 투수 7명
2019 / pitcherInning: 신규 투수 13명
2019 / pitcherWin: 신규 투수 8명
2019 / pitcherSave: 신규 투수 35명
2019 / pitcherHold: 신규 투수 17명
2019 / pitcherStrikeOut: 신규 투수 0명
2019 / pitcherGameCount: 신규 투수 0명

===== 2020 시즌 투수 수집 시작 =====
2020 / pitcherWar: 신규 투수 50명
2020 / pitcherEra: 신규 투수 31명
2020 / pitcherWhip: 신규 투수 7명
2020 / pitcherInning: 신규 투수 14명
2020 / pitcherWin: 신규 투수 2명
2020 / pitcherSave: 신규 투수 35명
2020 / pitcherHold: 신규 투수 21명
2020 / pitcherStrikeOut: 신규 투수 0명
2020 / pitcherGameCount: 신규 투수 4명

===== 2021 시즌 투수 수집 시작 =====
2021 / pitcherWar: 신규 투수 50명
2021 / pitcherEra: 신규 투수 3

,ranking,playerId,playerName,playerImageUrl,weight,height,backNumber,isRetire,isPlayer,osId,...,pitcherPaKkRate,pitcherPaBbRate,pitcherWpa,pitcherWar,pitcherStart,pitcherPitchCount,isQualified,isKoreanPlayer,collect_year,sort_source
0,NaN,67313,브리검,https://sports-phinf.pstatic.net/player/kbo/de...,90,190,8,N,Y,3545411,...,21.1,6.0,3.095,5.66,None,None,False,False,2018,pitcherWar
1,NaN,65543,린드블럼,https://sports-phinf.pstatic.net/player/kbo/de...,105,195,34,Y,Y,258747,...,23.1,5.6,3.954,5.28,None,None,False,False,2018,pitcherWar
2,NaN,61240,니퍼트,https://sports-phinf.pstatic.net/player/kbo/de...,103,203,0,Y,Y,130970,...,21.6,5.1,1.759,4.82,None,None,False,False,2018,pitcherWar
3,NaN,77829,김광현,https://sports-phinf.pstatic.net/player/kbo/de...,88,188,29,N,Y,124164,...,23.5,5.4,3.285,4.59,None,None,False,False,2018,pitcherWar
4,NaN,68135,윌슨,https://sports-phinf.pstatic.net/player/kbo/de...,84,188,35,Y,Y,3389641,...,21.6,5.1,4.288,4.54,None,None,False,False,2018,pitcherWar


In [24]:
selected_pitcher_cols = [
    "collect_year",
    "playerId",
    "playerName",
    "teamName",
    "isRetire",
    "isPlayer",

    "pitcherGameCount",
    "pitcherInning",

    "pitcherEra",
    "pitcherWhip",
    "pitcherWin",
    "pitcherLose",
    "pitcherSave",
    "pitcherHold",
    "pitcherStrikeOut",
    "pitcherBaseOnBalls",
    "pitcherHit",
    "pitcherHomeRun",
    "pitcherWar",

    "sort_source"
]

selected_pitcher_cols = [col for col in selected_pitcher_cols if col in pitcher_raw.columns]

pitcher_selected = pitcher_raw[selected_pitcher_cols].copy()

pitcher_selected.to_csv(
    "../data/pitcher_season_stats_2018_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료: ../data/pitcher_season_stats_2018_2026.csv")
print("데이터 크기:", pitcher_selected.shape)

display(pitcher_selected.head())

저장 완료: ../data/pitcher_season_stats_2018_2026.csv
데이터 크기: (1452, 17)


,collect_year,playerId,playerName,teamName,isRetire,isPlayer,pitcherGameCount,pitcherInning,pitcherEra,pitcherWhip,pitcherWin,pitcherLose,pitcherSave,pitcherHold,pitcherHit,pitcherWar,sort_source
0,2018,67313,브리검,넥센,N,Y,31,199,3.84422,1.20,11,7,0,1,188,5.66,pitcherWar
1,2018,65543,린드블럼,두산,Y,Y,26,168 2/3,2.88142,1.07,15,4,0,0,142,5.28,pitcherWar
2,2018,61240,니퍼트,KT,Y,Y,29,175 2/3,4.25237,1.41,8,8,0,0,209,4.82,pitcherWar
3,2018,77829,김광현,SK,N,Y,25,136,2.97794,1.14,11,8,0,0,125,4.59,pitcherWar
4,2018,68135,윌슨,LG,Y,Y,26,170,3.07059,1.14,9,4,0,0,158,4.54,pitcherWar
